# PET, written out: one forward pass and one Adam step

**Every learned PET layer and the complete forward pass are defined in this
notebook.** The only runtime library is **PyTorch** (plus Python's standard
library). There are no imports from metatrain, metatomic, or metatensor, and no
inherited PET implementation, dynamic source loading, or hidden helper module.

Keep this notebook beside `pet_input.pt` and run all cells with a PyTorch kernel.
You can copy these two files out of the repository. The `.pt` file contains the
resolved `options-pet-oam-l-modern-mptrj-salex-direct-epoch@50.yaml` config and one
real MPtrj batch, already prepared. No data preprocessing runs here.

This is the configured **relative-geometry, feedforward, RMSNorm, SwiGLU, PreLN**
PET with direct energy/force/stress heads, without long-range or ZBL terms.
Unused architecture variants, export wrappers, and dataset plumbing have been
removed to make this path readable. Fresh weights are initialized; there is one
forward, one backward, and one Adam update, with no training loop.


In [1]:
from pathlib import Path
from math import prod
import time

import torch
from torch import nn
import torch.nn.functional as F

# Works from the notebook directory, or from the repository root.
INPUT_PATH = Path("pet_input.pt")
if not INPUT_PATH.is_file():
    INPUT_PATH = Path("notebooks/pet_input.pt")
payload = torch.load(INPUT_PATH, map_location="cpu", weights_only=True)
assert payload["format_version"] == 2, "Regenerate the input with the companion exporter."
config = payload["config"]
hypers = payload["architecture"]["model"]
train_config = payload["architecture"]["training"]
target_info = payload["dataset_info"]["targets"]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
assert config["base_precision"] == 32
torch.set_num_threads(4)
torch.manual_seed(config["seed"])
if device.type == "cuda":
    torch.cuda.manual_seed_all(config["seed"])
    torch.cuda.init()
    torch.cuda.reset_peak_memory_stats(device)
print("Device:", device, "| dtype:", dtype, "| PyTorch:", torch.__version__)
print("Config:", payload["provenance"]["config_name"])
print("Frozen target statistics:", payload["provenance"]["statistics_checkpoint"])
print("Model configuration:", hypers)


/home/ryoji/miniconda3/envs/pet-notebook/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Device: cuda:0 | dtype: torch.float32 | PyTorch: 2.12.1+cu130
Config: options-pet-oam-l-modern-mptrj-salex-direct-epoch@50.yaml
Frozen target statistics: pet-oam-l-modern-mptrj-salex-direct-ddp.ckpt
Model configuration: {'cutoff': 10.0, 'num_neighbors_adaptive': 40, 'adaptive_cutoff_method': 'grid', 'neighbor_cell_shift_mode': 'all', 'geometry_mode': 'relative', 'cutoff_function': 'Bump', 'cutoff_width': 0.5, 'd_pet': 512, 'd_head': 512, 'd_node': 2048, 'd_feedforward': 1024, 'num_heads': 8, 'num_attention_layers': 2, 'num_gnn_layers': 3, 'normalization': 'RMSNorm', 'activation': 'SwiGLU', 'attention_temperature': 1.0, 'transformer_type': 'PreLN', 'featurizer_type': 'feedforward', 'zbl': False, 'long_range': {'enable': False, 'use_ewald': False, 'smearing': 1.4, 'kspace_resolution': 1.33, 'interpolation_nodes': 5}}


## Already prepared input

`B` is the number of structures, `N` the number of atoms, and `K` the padded
neighborhood size. The saved batch has **B=16, N=512, K=52**. The configured
40 neighbors is an adaptive-cutoff parameter, not a fixed token count.

| Tensor | Shape | Meaning |
|---|---|---|
| `element_indices_nodes` | `N` | Central species indices in the 89-element vocabulary |
| `element_indices_neighbors` | `N × K` | Neighbor species indices |
| `edge_vectors` | `N × K × 3` | Cartesian displacements including periodic images |
| `edge_distances` | `N × K` | Edge lengths |
| `cutoff_factors` | `N × K` | Smooth distance weights |
| `padding_mask` | `N × K` | **True means a real edge**, False means padding |
| `reverse_neighbor_index` | `N*K` | Flat index of the reverse edge for message exchange |
| `system_indices` | `N` | Structure index for each atom |
| `cells`, `num_atoms` | `B × 3 × 3`, `B` | Cell matrices for stress and counts for energy loss |

Positions and periodic image positions are also saved for reference, but this
config's learned geometry features use relative vectors and distances. Rotations,
neighbor lists, adaptive cutoffs, baseline subtraction, and label scaling were
computed outside this notebook. The input geometry is frozen: forces/stress here
come from direct heads, not derivatives of energy with respect to coordinates.


In [2]:
inputs = {name: value.to(device) for name, value in payload["inputs"].items()}
system_indices = payload["system_indices"].to(device)
cells = payload["cells"].to(device=device, dtype=dtype)
num_atoms = payload["num_atoms"].to(device=device, dtype=dtype)
targets = {name: value.to(device) for name, value in payload["loss_targets"].items()}
prediction_scales = {name: value.to(device) for name, value in payload["prediction_scales"].items()}

assert all(not value.requires_grad for value in inputs.values())
assert int(num_atoms.sum()) <= train_config["max_atoms_per_batch"]
assert torch.equal(torch.bincount(system_indices), num_atoms.long())
assert torch.equal(payload["sample_labels"][:, 0], payload["system_indices"].int())
for name, spec in target_info.items():
    expected_samples = (
        payload["sample_labels"] if spec["sample_kind"] == "atom"
        else torch.arange(len(cells), dtype=torch.int32).reshape(-1, 1)
    )
    assert torch.equal(payload["target_samples"][name], expected_samples), name
    assert list(targets[name].shape[1:]) == spec["shape"]
    assert torch.isfinite(targets[name]).all()
print(f"B={len(cells)}, N={int(num_atoms.sum())}, K={inputs['padding_mask'].shape[1]}")
print("Dataset indices:", payload["provenance"]["dataset_indices"])
for name, value in inputs.items():
    print(f"{name:28s} {str(tuple(value.shape)):18s} {value.dtype}")


B=16, N=512, K=52
Dataset indices: [527903, 739617, 511522, 305766, 111102, 781751, 1146343, 1284170, 436703, 796520, 1572617, 247362, 179354, 736149, 1222165, 18790]
element_indices_nodes        (512,)             torch.int64
element_indices_neighbors    (512, 52)          torch.int64
edge_vectors                 (512, 52, 3)       torch.float32
edge_distances               (512, 52)          torch.float32
padding_mask                 (512, 52)          torch.bool
reverse_neighbor_index       (512, 52)          torch.int64
cutoff_factors               (512, 52)          torch.float32
node_positions               (512, 3)           torch.float32
neighbor_image_positions     (512, 52, 3)       torch.float32


## 1. Gated feedforward network and attention

The repository's branch named `SwiGLU` computes **`value * sigmoid(gate)`** after
a single projection to twice the hidden width, followed by an output projection.
That exact formula is retained below. The readout heads later use `SiLU`.

Attention projects tokens into Q, K, V and computes

`softmax(Q @ K.T / (sqrt(head_dim) * temperature) + log(cutoff_weight)) @ V`.

Cutoff weights apply to **key** tokens. Padding receives zero weight before the
same `1e-15` clamp used in PET. `manual_attention` spells out the entire operation;
the default uses PyTorch's fused equivalent, as the original direct-head model
does. Set `USE_MANUAL_ATTENTION = True` at model construction to use the visible
matmul/softmax implementation instead.


In [3]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward):
        super().__init__()
        self.w_in = nn.Linear(d_model, 2 * dim_feedforward)
        self.w_out = nn.Linear(dim_feedforward, d_model)

    def forward(self, x):
        value, gate = self.w_in(x).chunk(2, dim=-1)
        return self.w_out(value * torch.sigmoid(gate))


def manual_attention(q, k, v, attention_bias, temperature):
    scores = q @ k.transpose(-2, -1)
    scores = scores / (k.shape[-1] ** 0.5 * temperature) + attention_bias
    return scores.softmax(dim=-1) @ v


class AttentionBlock(nn.Module):
    def __init__(self, total_dim, num_heads, temperature):
        super().__init__()
        assert total_dim % num_heads == 0
        self.input_linear = nn.Linear(total_dim, 3 * total_dim)
        self.output_linear = nn.Linear(total_dim, total_dim)
        self.num_heads = num_heads
        self.head_dim = total_dim // num_heads
        self.temperature = temperature

    def forward(self, x, cutoff_factors, use_manual_attention=False):
        # x: [N, 1+K, d_pet]; each central atom defines one attention sequence.
        n_atoms, n_tokens, width = x.shape
        qkv = self.input_linear(x)
        qkv = qkv.reshape(n_atoms, n_tokens, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # [N, heads, 1+K, head_dim]
        bias = cutoff_factors[:, None, :, :].clamp(min=1e-15).log()
        if use_manual_attention:
            attended = manual_attention(q, k, v, bias, self.temperature)
        else:
            attended = F.scaled_dot_product_attention(
                q, k, v, attn_mask=bias,
                scale=1.0 / (self.head_dim ** 0.5 * self.temperature),
            )
        attended = attended.transpose(1, 2).reshape(n_atoms, n_tokens, width)
        return self.output_linear(attended)


## 2. PreLN transformer layer

Central nodes have width **2048**; edge tokens and attention have width **512**.
A learned contraction brings the central token into attention, and a learned
expansion brings its update back. Node and edge updates each have residual
connections and their own feedforward networks. Each GNN layer contains two of
these transformer layers.

`nn.RMSNorm` normalizes by the root mean square of the last feature dimension
(with its epsilon and learned scale). It is applied **before** each sublayer in
this PreLN path. The edge-combination block between GNN layers uses `LayerNorm`,
matching the repository even though attention normalization is RMSNorm.


In [4]:
class TransformerLayer(nn.Module):
    def __init__(self, h):
        super().__init__()
        d, d_node = h["d_pet"], h["d_node"]
        self.attention = AttentionBlock(d, h["num_heads"], h["attention_temperature"])
        self.norm_attention = nn.RMSNorm(d)
        self.norm_mlp = nn.RMSNorm(d)
        self.mlp = FeedForward(d, h["d_feedforward"])
        self.center_contraction = nn.Linear(d_node, d)
        self.center_expansion = nn.Linear(d, d_node)
        self.norm_center_features = nn.RMSNorm(d_node)
        self.center_mlp = FeedForward(d_node, 2 * d_node)

    def forward(self, node, edge, cutoff_factors, use_manual_attention=False):
        # node: [N, 1, d_node], edge: [N, K, d_pet]
        central_token = self.center_contraction(node)
        tokens = torch.cat([central_token, edge], dim=1)
        update = self.attention(
            self.norm_attention(tokens), cutoff_factors, use_manual_attention
        )
        node_update, edge_update = torch.split(update, [1, edge.shape[1]], dim=1)
        node = node + self.center_expansion(node_update)
        node = node + self.center_mlp(self.norm_center_features(node))
        edge = edge + edge_update
        edge = edge + self.mlp(self.norm_mlp(edge))
        return node, edge


class Transformer(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerLayer(h) for _ in range(h["num_attention_layers"])
        ])

    def forward(self, node, edge, cutoff_factors, use_manual_attention=False):
        for layer in self.layers:
            node, edge = layer(node, edge, cutoff_factors, use_manual_attention)
        return node, edge


## 3. Cartesian geometry and neighborhood tokens

Each edge starts with four numbers: **dx, dy, dz, distance**. Their learned
projection is concatenated with the incoming edge message. From the second GNN
layer onward a fresh neighbor-species embedding is also included. A small MLP
compresses the concatenation to width 512.

The central token is prepended to its neighborhood. Its cutoff weight is 1;
neighbor tokens use their saved smooth cutoff weights. This produces 53 tokens
per attention sequence for this batch: one central token and 52 neighbor slots.


In [5]:
class CartesianTransformer(nn.Module):
    def __init__(self, h, num_species, is_first):
        super().__init__()
        self.is_first = is_first
        d = h["d_pet"]
        self.trans = Transformer(h)
        self.edge_embedder = nn.Linear(4, d)
        n_merge = 2 if is_first else 3
        self.compress = nn.Sequential(
            nn.Linear(n_merge * d, d), nn.SiLU(), nn.Linear(d, d)
        )
        self.neighbor_embedder = (
            nn.Identity() if is_first else nn.Embedding(num_species, d)
        )

    def forward(self, node, incoming_edge, inputs, use_manual_attention=False):
        geometry = torch.cat([
            inputs["edge_vectors"], inputs["edge_distances"][..., None]
        ], dim=2)
        embedded_geometry = self.edge_embedder(geometry)
        if self.is_first:
            edge_tokens = torch.cat([embedded_geometry, incoming_edge], dim=2)
        else:
            species = self.neighbor_embedder(inputs["element_indices_neighbors"])
            edge_tokens = torch.cat([embedded_geometry, species, incoming_edge], dim=2)
        edge_tokens = self.compress(edge_tokens)

        # Every query sees the same distance weights for the key tokens.
        central_weight = torch.ones_like(inputs["cutoff_factors"][:, :1])
        edge_weights = inputs["cutoff_factors"].masked_fill(~inputs["padding_mask"], 0.0)
        key_weights = torch.cat([central_weight, edge_weights], dim=1)
        cutoff_matrix = key_weights[:, None, :].repeat(1, key_weights.shape[1], 1)
        node, edge = self.trans(
            node[:, None, :], edge_tokens, cutoff_matrix, use_manual_attention
        )
        return node.squeeze(1), edge


## 4. Full PET: embeddings, message exchange, and all output heads

The `PET` below inherits only from **`torch.nn.Module`**. Its constructor builds
all trainable parts. Its `forward` is the entire path from saved tensors to
predictions; there are no calls to external PET helpers.

After each Cartesian transformer, the reverse index gathers the message on
**j → i** for every **i → j** edge. The update is

`edge_next = edge_previous + edge_output + MLP(LayerNorm([edge_output, reverse_edge]))`.

The feedforward architecture uses the final GNN features for readout. Each target
has separate node and edge heads (`Linear → SiLU → Linear → SiLU → Linear`).
The final widths are 1 for energy, 3 for force, and 9 for stress. Padded edges
are masked, real edge contributions are cutoff-weighted, and neighbors are summed.

Energy sums atom contributions within each structure. Force stays per atom.
Stress reshapes each atomic contribution to 3×3, divides by cell volume,
symmetrizes it, then sums over atoms. These direct heads do not differentiate
energy to obtain force/stress.


In [6]:
class PET(nn.Module):
    def __init__(self, h, atomic_types, targets, use_manual_attention=False):
        super().__init__()
        # This notebook implements precisely the branch used by the saved YAML.
        required = {
            "geometry_mode": "relative", "featurizer_type": "feedforward",
            "normalization": "RMSNorm", "activation": "SwiGLU",
            "transformer_type": "PreLN",
        }
        for key, expected in required.items():
            if h[key] != expected:
                raise ValueError(f"This notebook implements {key}={expected!r}.")
        assert not h["zbl"] and not h["long_range"]["enable"]
        assert h["d_node"] != h["d_pet"]  # Configured central expansion path.
        assert set(targets) == {"energy", "non_conservative_force", "non_conservative_stress"}
        self.use_manual_attention = use_manual_attention
        self.target_info = targets
        self.output_shapes = {name: spec["shape"] for name, spec in targets.items()}
        d, d_node, d_head = h["d_pet"], h["d_node"], h["d_head"]
        num_species = len(atomic_types)
        self.gnn_layers = nn.ModuleList([
            CartesianTransformer(h, num_species, is_first=(i == 0))
            for i in range(h["num_gnn_layers"])
        ])
        self.combination_norms = nn.ModuleList([
            nn.LayerNorm(2 * d) for _ in self.gnn_layers
        ])
        self.combination_mlps = nn.ModuleList([
            nn.Sequential(nn.Linear(2 * d, 2 * d), nn.SiLU(), nn.Linear(2 * d, d))
            for _ in self.gnn_layers
        ])
        self.node_embedders = nn.ModuleList([nn.Embedding(num_species, d_node)])
        self.edge_embedder = nn.Embedding(num_species, d)
        self.node_heads = nn.ModuleDict()
        self.edge_heads = nn.ModuleDict()
        self.node_last_layers = nn.ModuleDict()
        self.edge_last_layers = nn.ModuleDict()
        for name, shape in self.output_shapes.items():
            self.node_heads[name] = nn.Sequential(
                nn.Linear(d_node, d_head), nn.SiLU(),
                nn.Linear(d_head, d_head), nn.SiLU(),
            )
            self.edge_heads[name] = nn.Sequential(
                nn.Linear(d, d_head), nn.SiLU(),
                nn.Linear(d_head, d_head), nn.SiLU(),
            )
            self.node_last_layers[name] = nn.Linear(d_head, prod(shape))
            self.edge_last_layers[name] = nn.Linear(d_head, prod(shape))

    def forward(self, inputs, system_indices, cells):
        # 1. Initial species embeddings.
        node = self.node_embedders[0](inputs["element_indices_nodes"])
        edge = self.edge_embedder(inputs["element_indices_neighbors"])

        # 2. Neighborhood attention followed by message exchange between atoms.
        for gnn, norm, mlp in zip(
            self.gnn_layers, self.combination_norms, self.combination_mlps, strict=True
        ):
            node, edge_out = gnn(node, edge, inputs, self.use_manual_attention)
            reverse_edge = edge_out.flatten(0, 1)[inputs["reverse_neighbor_index"]]
            reverse_edge = reverse_edge.reshape_as(edge_out)
            combined = torch.cat([edge_out, reverse_edge], dim=-1)
            edge = edge + edge_out + mlp(norm(combined))

        # 3. Separate node and edge heads for all three physical targets.
        predictions = {}
        for name, shape in self.output_shapes.items():
            node_prediction = self.node_last_layers[name](self.node_heads[name](node))
            edge_prediction = self.edge_last_layers[name](self.edge_heads[name](edge))
            edge_prediction = torch.where(
                inputs["padding_mask"][..., None], edge_prediction, 0.0
            )
            edge_sum = (edge_prediction * inputs["cutoff_factors"][..., None]).sum(dim=1)
            atomic = (node_prediction + edge_sum).reshape(-1, *shape)

            # 4. Convert the atomic stress tensor into a symmetric density.
            if name == "non_conservative_stress":
                volumes = torch.stack([torch.abs(torch.det(cell)) for cell in cells])
                volumes = torch.where(volumes == 0.0, torch.inf, volumes)
                atomic = atomic / volumes[system_indices, None, None, None]
                atomic = (atomic + atomic.transpose(1, 2)) / 2.0

            # 5. Energy/stress aggregate per structure; force remains per atom.
            if self.target_info[name]["sample_kind"] == "system":
                total = atomic.new_zeros((len(cells), *shape))
                total.index_add_(0, system_indices, atomic)
                predictions[name] = total
            else:
                predictions[name] = atomic
        return predictions


In [7]:
USE_MANUAL_ATTENTION = False
model = PET(
    hypers, payload["dataset_info"]["atomic_types"], target_info,
    use_manual_attention=USE_MANUAL_ATTENTION,
).to(device=device, dtype=dtype)
model.train()
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {parameter_count:,}")
print("GNN layers:", len(model.gnn_layers))
print("Attention layers per GNN:", len(model.gnn_layers[0].trans.layers))
print("Fused attention:", not USE_MANUAL_ATTENTION)
for name in model.output_shapes:
    print(name, "node head:", model.node_heads[name])
    print(name, "last layer:", model.node_last_layers[name])


Trainable parameters: 192,896,538
GNN layers: 3
Attention layers per GNN: 2
Fused attention: True
energy node head: Sequential(
  (0): Linear(in_features=2048, out_features=512, bias=True)
  (1): SiLU()
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): SiLU()
)
energy last layer: Linear(in_features=512, out_features=1, bias=True)
non_conservative_force node head: Sequential(
  (0): Linear(in_features=2048, out_features=512, bias=True)
  (1): SiLU()
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): SiLU()
)
non_conservative_force last layer: Linear(in_features=512, out_features=3, bias=True)
non_conservative_stress node head: Sequential(
  (0): Linear(in_features=2048, out_features=512, bias=True)
  (1): SiLU()
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): SiLU()
)
non_conservative_stress last layer: Linear(in_features=512, out_features=9, bias=True)


## 5. One forward pass, with tensor shapes

Hooks report shapes from the same forward pass. They do not run the model again
or retain copies of activations. Gradients are enabled so the following optimizer
cell can use this computation graph. Outputs are ordinary PyTorch tensors.


In [8]:
assert train_config["weight_decay"] is None  # PET selects Adam for this setting.
optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])
optimizer.zero_grad(set_to_none=True)
trace = []


def record_shapes(name):
    def hook(module, args, output):
        values = output if isinstance(output, tuple) else (output,)
        trace.append((name, tuple(args[0].shape), [tuple(t.shape) for t in values]))
    return hook


handles = [
    module.register_forward_hook(record_shapes(name))
    for name, module in model.named_modules()
    if isinstance(module, (CartesianTransformer, AttentionBlock))
]
if device.type == "cuda":
    torch.cuda.synchronize()
started = time.perf_counter()
try:
    predictions = model(inputs, system_indices, cells)
finally:
    for handle in handles:
        handle.remove()
if device.type == "cuda":
    torch.cuda.synchronize()
print(f"Forward time: {time.perf_counter() - started:.3f} seconds")
for name, shape_in, shapes_out in trace:
    print(f"{name}: {shape_in} -> {shapes_out}")
assert len(trace) == hypers["num_gnn_layers"] * (1 + hypers["num_attention_layers"])
for name, value in predictions.items():
    assert torch.isfinite(value).all(), name
    print(f"{name}: {tuple(value.shape)}")
    print("  first values:", value.detach().flatten()[:6].cpu().tolist())
stress = predictions["non_conservative_stress"]
assert torch.allclose(stress, stress.transpose(1, 2), atol=1e-6)


Forward time: 0.396 seconds
gnn_layers.0.trans.layers.0.attention: (512, 53, 512) -> [(512, 53, 512)]
gnn_layers.0.trans.layers.1.attention: (512, 53, 512) -> [(512, 53, 512)]
gnn_layers.0: (512, 2048) -> [(512, 2048), (512, 52, 512)]
gnn_layers.1.trans.layers.0.attention: (512, 53, 512) -> [(512, 53, 512)]
gnn_layers.1.trans.layers.1.attention: (512, 53, 512) -> [(512, 53, 512)]
gnn_layers.1: (512, 2048) -> [(512, 2048), (512, 52, 512)]
gnn_layers.2.trans.layers.0.attention: (512, 53, 512) -> [(512, 53, 512)]
gnn_layers.2.trans.layers.1.attention: (512, 53, 512) -> [(512, 53, 512)]
gnn_layers.2: (512, 2048) -> [(512, 2048), (512, 52, 512)]
energy: (16, 1)
  first values: [187.75814819335938, 75.28852844238281, 202.04531860351562, 14.501835823059082, 74.08098602294922, 62.825626373291016]
non_conservative_force: (512, 3, 1)
  first values: [-0.05688824504613876, -2.211146354675293, -3.6474967002868652, -0.1188468486070633, -1.9070780277252197, -3.292142152786255]
non_conservative_stres

## 6. Configured loss

The saved targets have already had composition offsets and per-target scales
removed. Energy labels are already divided by atom count. Match that loss space
by dividing predicted energy by atom count and applying the saved per-property
multipliers. Force stays per atom; stress stays per structure.

| Target | Huber delta | Weight |
|---|---:|---:|
| Energy per atom | 0.015 | 1.0 |
| Direct force | 0.04 | 1.0 |
| Direct stress | 0.004 | 0.01 |

Each term is mean-reduced over its components. Huber's quadratic branch is
`0.5 * error**2`; its linear branch is `delta * (abs(error) - 0.5 * delta)`.
Predictions and labels are in normalized residual space, not physical-unit
inference outputs. Frozen statistics came from the matching PET checkpoint
listed above; none of its learned neural-network weights are used.


In [9]:
loss_terms = {}
loss_predictions = {}
for name, value in predictions.items():
    assert value.shape == targets[name].shape
    if (target_info[name]["sample_kind"] == "system"
            and name not in train_config["per_structure_targets"]):
        value = value / num_atoms.reshape(-1, *([1] * (value.ndim - 1)))
    value = value * prediction_scales[name]
    loss_predictions[name] = value
    spec = train_config["loss"][name]
    assert spec["type"] == "huber"
    loss_terms[name] = spec["weight"] * F.huber_loss(
        value, targets[name], delta=spec["delta"], reduction=spec["reduction"]
    )
loss = sum(loss_terms.values())
assert torch.isfinite(loss)
for name, term in loss_terms.items():
    print(f"{name:26s} weighted loss = {term.item():.8f}")
print(f"Total loss = {loss.item():.8f}")


energy                     weighted loss = 0.03071209
non_conservative_force     weighted loss = 0.07877859
non_conservative_stress    weighted loss = 0.00001287
Total loss = 0.10950355


## 7. One backward pass and one Adam update

Clip the gradient norm to **1.0**, then update at the configured base learning rate
**1e-4**. The 50-epoch warmup/cosine schedule is omitted here: its initial zero
learning rate would leave the parameters unchanged. This demonstrates one
optimizer update rather than replaying training step zero.

The checks show that all three heads receive gradients and a watched parameter
changes. Rerun from model construction for a fresh experiment; rerunning only
this cell would attempt to reuse the consumed autograd graph.


In [10]:
watched_name = "node_last_layers.energy.weight"
watched_parameter = dict(model.named_parameters())[watched_name]
before = watched_parameter.detach().clone()
loss.backward()
for target_name in targets:
    head_parameters = [
        *model.node_last_layers[target_name].parameters(),
        *model.edge_last_layers[target_name].parameters(),
    ]
    head_grad = sum(p.grad.detach().square().sum() for p in head_parameters).sqrt()
    assert torch.isfinite(head_grad) and head_grad > 0, target_name
    print(f"{target_name:26s} head gradient norm = {head_grad.item():.6g}")
grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=train_config["grad_clip_norm"], error_if_nonfinite=True
)
optimizer.step()  # The only optimizer update in this notebook.
update = (watched_parameter.detach() - before).abs().max().item()
assert update > 0
assert int(optimizer.state[watched_parameter]["step"].item()) == 1
print(f"Gradient norm before clipping: {grad_norm.item():.6f}")
print(f"Adam learning rate: {optimizer.param_groups[0]['lr']}")
print(f"Updated {watched_name}: max absolute change = {update:.8g}")
if device.type == "cuda":
    print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated(device) / 2**30:.2f} GiB")


energy                     head gradient norm = 1.14805
non_conservative_force     head gradient norm = 1.50146
non_conservative_stress    head gradient norm = 4.25202e-05
Gradient norm before clipping: 4.598694
Adam learning rate: 0.0001
Updated node_last_layers.energy.weight: max absolute change = 9.9997967e-05
Peak allocated GPU memory: 8.99 GiB


`inputs`, `model`, `predictions`, `loss_terms`, and `trace` remain available for
inspection. Edit any layer above to experiment. No model checkpoint is written.

Only regenerating the prepared batch requires the original repository and data:
run `python scripts/prepare_pet_notebook_input.py` from the repository root.
Geometry or neighbor-rule changes require regenerating the input. The saved
config describes the architecture and frozen preprocessing used by this batch.

The inline implementation was adapted from `src/metatrain/pet/model.py` and
`src/metatrain/pet/modules/transformer.py`. Parameter counts, predictions, losses,
and parameter gradients are checked against that implementation outside the
notebook. The source license is reproduced below so these two files can be used
independently of the checkout.


<details>
<summary>Source license (BSD 3-Clause)</summary>

```text
BSD 3-Clause License

Copyright (c) 2023, Laboratory of Computational Science and Modeling

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
```
</details>
